# Bước 2: Xây Dựng Ordinal Network

**Mục tiêu:** Mã hoá chuỗi log-return thành ordinal patterns, tìm optimal (d, τ) bằng grid search, xây ma trận chuyển tiếp và đồ thị mạng ordinal.

**Input:** `data/preprocessed_log_returns.csv`  
**Output:** `data/ordinal_patterns.pkl`, `data/transition_matrices.pkl`, `data/optimal_params.csv`

In [1]:
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "networkx", "scipy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("✓ Packages ready.")

✓ Packages ready.


In [2]:
import warnings; warnings.filterwarnings("ignore")
import itertools, pickle
import sys
from pathlib import Path
from math import factorial

import warnings; warnings.filterwarnings("ignore")

# Force Agg backend before any pyplot import
import matplotlib
import matplotlib.backends
matplotlib.use('Agg')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx
from scipy.stats import entropy as sp_entropy

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})
DATA_DIR = Path("data")

# ── Load preprocessed log-returns ────────────────────────────────────────────
lr_path = DATA_DIR / "preprocessed_log_returns.csv"
if not lr_path.exists():
    raise FileNotFoundError("Run 01_preprocessing.ipynb first!")

log_ret = pd.read_csv(lr_path, index_col=0, parse_dates=True)
COINS = list(log_ret.columns)
T, N = log_ret.shape
print(f"Log-returns shape : {log_ret.shape}")
print(f"Coins             : {COINS}")
print(f"Date range        : {log_ret.index[0].date()} → {log_ret.index[-1].date()}")

Log-returns shape : (2210, 8)
Coins             : ['ADA', 'BNB', 'BTC', 'DOGE', 'ETH', 'LTC', 'SOL', 'XRP']
Date range        : 2020-04-11 → 2026-04-29


## 1 · Ordinal Pattern Encoding

Với embedding dimension $d$ và delay $\tau$, mỗi window $[x_t, x_{t+\tau}, \ldots, x_{t+(d-1)\tau}]$ được mã hoá thành **ordinal pattern** (hoán vị thứ hạng).  
Số pattern tối đa: $d!$

In [4]:
def encode_ordinal_patterns(series: np.ndarray, d: int, tau: int) -> np.ndarray:
    """
    Encode a 1-D time series into ordinal patterns.

    Parameters
    ----------
    series : 1-D array of floats
    d      : embedding dimension (pattern length)
    tau    : time delay

    Returns
    -------
    patterns : array of integer tuples (as tuple indices) length T - (d-1)*tau
    """
    n = len(series)
    n_patterns = n - (d - 1) * tau
    if n_patterns <= 0:
        return np.empty((0, d), dtype=np.int64)

    patterns = []
    for i in range(n_patterns):
        window = series[i: i + d * tau: tau]   # length d with delay tau
        rank = list(np.argsort(np.argsort(window)).astype(int))  # ordinal rank
        patterns.append(rank)
    return np.array(patterns, dtype=np.int64)


def permutation_entropy(patterns, d: int, normalize: bool = True) -> float:
    """Shannon entropy of ordinal pattern distribution."""
    if len(patterns) is None or len(patterns) == 0:
        return np.nan
    arr = np.asarray(patterns, dtype=np.int64)
    unique, counts = np.unique(arr, axis=0, return_counts=True)
    probs = counts / counts.sum()
    H = -np.sum(probs * np.log(probs + 1e-12))
    if normalize:
        max_H = np.log(factorial(d))
        return H / max_H if max_H > 0 else 0.0
    return H


# Quick sanity check
test_series = np.array([3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0])
test_pats = encode_ordinal_patterns(test_series, d=3, tau=1)
print("Test patterns (d=3, tau=1):", test_pats)
print("PE:", round(permutation_entropy(test_pats, d=3), 4))

Test patterns (d=3, tau=1): [[1 0 2]
 [0 2 1]
 [1 0 2]
 [0 1 2]
 [1 2 0]
 [2 0 1]]
PE: 0.871


## 2 · Grid Search tìm optimal (d, τ)

**Tiêu chí:** Tối thiểu hoá phương sai PE theo thời gian (cross-validation stability) và tối đa hoá sự phân biệt giữa các đồng tiền. Tránh overfitting bằng cách dùng rolling estimation.

In [5]:
D_RANGE   = [3, 4, 5, 6]
TAU_RANGE = [1, 2, 3, 5]
ROLL_W    = 252   # 1-year rolling window for stability assessment

grid_results = []

for coin in COINS:
    series = log_ret[coin].values
    for d in D_RANGE:
        for tau in TAU_RANGE:
            min_len = (d - 1) * tau + 1
            if len(series) < ROLL_W + min_len:
                continue

            # Rolling PE across 1-year windows (step=30 days)
            pe_vals = []
            for start in range(0, len(series) - ROLL_W - min_len, 30):
                window = series[start: start + ROLL_W]
                pats = encode_ordinal_patterns(window, d, tau)
                if len(pats) > 0:
                    pe_vals.append(permutation_entropy(pats, d))

            if len(pe_vals) < 3:
                continue

            grid_results.append({
                "coin": coin,
                "d": d,
                "tau": tau,
                "mean_PE": np.mean(pe_vals),
                "std_PE":  np.std(pe_vals),   # stability: lower is better
                "cv_PE":   np.std(pe_vals) / (np.mean(pe_vals) + 1e-12),
                "n_patterns": factorial(d),
            })

grid_df = pd.DataFrame(grid_results)
print(f"Grid search results: {len(grid_df)} rows")
display(grid_df.groupby(["d", "tau"])[["mean_PE", "std_PE", "cv_PE"]].mean().round(4))

Grid search results: 128 rows


mean_PE  std_PE   cv_PE
d tau                         
3 1     0.9964  0.0026  0.0026
  2     0.9965  0.0024  0.0024
  3     0.9962  0.0032  0.0032
  5     0.9961  0.0029  0.0029
4 1     0.9870  0.0045  0.0045
  2     0.9866  0.0046  0.0046
  3     0.9872  0.0043  0.0044
  5     0.9861  0.0048  0.0049
5 1     0.9445  0.0078  0.0082
  2     0.9431  0.0080  0.0085
  3     0.9433  0.0083  0.0088
  5     0.9402  0.0084  0.0089
6 1     0.8042  0.0053  0.0066
  2     0.8017  0.0055  0.0069
  3     0.7991  0.0055  0.0069
  5     0.7938  0.0048  0.0061

In [6]:
# Heatmap: mean PE across all coins for each (d, tau)
pivot_pe  = grid_df.groupby(["d", "tau"])["mean_PE"].mean().unstack("tau")
pivot_cv  = grid_df.groupby(["d", "tau"])["cv_PE"].mean().unstack("tau")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(pivot_pe, annot=True, fmt=".3f", cmap="YlOrRd", ax=ax1)
ax1.set_title("Mean Permutation Entropy (higher = more random)", fontsize=11)
sns.heatmap(pivot_cv, annot=True, fmt=".3f", cmap="YlGnBu_r", ax=ax2)
ax2.set_title("CV of Rolling PE (lower = more stable)", fontsize=11)
plt.suptitle("Grid Search: (d, τ) Parameter Selection", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_gridsearch_dtau.png", bbox_inches="tight")
plt.show()

# Select optimal: balance high PE discrimination and low CV
# Score = mean_PE / cv_PE  (want high PE with low variance)
grid_df["score"] = grid_df["mean_PE"] / (grid_df["cv_PE"] + 0.01)
best_per_coin = grid_df.loc[grid_df.groupby("coin")["score"].idxmax()].reset_index(drop=True)

print("\n=== Optimal (d, τ) per coin ===")
display(best_per_coin[["coin", "d", "tau", "mean_PE", "std_PE", "score"]])

# Global consensus
mode_d   = int(best_per_coin["d"].mode()[0])
mode_tau = int(best_per_coin["tau"].mode()[0])
print(f"\n→ Global consensus: d={mode_d}, τ={mode_tau}")

BEST_D, BEST_TAU = mode_d, mode_tau


=== Optimal (d, τ) per coin ===


,coin,d,tau,mean_PE,std_PE,score
0,ADA,3,5,0.996137,0.002462,79.874109
1,BNB,3,5,0.996627,0.002317,80.864480
2,BTC,3,2,0.997192,0.001833,84.237897
3,DOGE,3,1,0.997517,0.001543,86.387019
4,ETH,3,1,0.997104,0.001575,86.108246
5,LTC,3,3,0.996373,0.002667,78.596382
6,SOL,3,1,0.997116,0.002042,82.763695
7,XRP,3,2,0.996882,0.001809,84.377997



→ Global consensus: d=3, τ=1


## 3 · Ma trận chuyển tiếp (Transition Matrix)

$T_{ij} = P(\pi_{t+1} = j \mid \pi_t = i)$ — xác suất chuyển từ pattern $i$ sang pattern $j$.

In [7]:
def build_transition_matrix(patterns: np.ndarray, d: int) -> tuple:
    """
    Build transition probability matrix from ordinal pattern sequence.

    Returns
    -------
    T_matrix : (d! x d!) numpy array of transition probabilities
    pat2idx  : dict mapping pattern tuple -> index
    idx2pat  : list of pattern tuples
    """
    all_perms = list(itertools.permutations(range(d)))
    n_perms   = len(all_perms)
    pat2idx   = {p: i for i, p in enumerate(all_perms)}

    count_matrix = np.zeros((n_perms, n_perms), dtype=float)

    for k in range(len(patterns) - 1):
        pi_now  = tuple(patterns[k])
        pi_next = tuple(patterns[k + 1])
        if pi_now in pat2idx and pi_next in pat2idx:
            count_matrix[pat2idx[pi_now], pat2idx[pi_next]] += 1

    # Row-normalise to get probabilities (Laplace smoothing to avoid zero rows)
    row_sums = count_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    T_matrix = count_matrix / row_sums

    return T_matrix, pat2idx, all_perms


# Build for each coin using global best (d, tau)
transition_matrices = {}
all_patterns        = {}

for coin in COINS:
    series   = log_ret[coin].values
    patterns = encode_ordinal_patterns(series, BEST_D, BEST_TAU)
    T, pat2idx, idx2pat = build_transition_matrix(patterns, BEST_D)
    transition_matrices[coin] = {
        "T": T, "pat2idx": pat2idx, "idx2pat": idx2pat,
        "d": BEST_D, "tau": BEST_TAU,
    }
    all_patterns[coin] = patterns
    print(f"{coin}: {len(patterns)} patterns,  T shape={T.shape}")

ADA: 2208 patterns,  T shape=(6, 6)
BNB: 2208 patterns,  T shape=(6, 6)
BTC: 2208 patterns,  T shape=(6, 6)
DOGE: 2208 patterns,  T shape=(6, 6)
ETH: 2208 patterns,  T shape=(6, 6)
LTC: 2208 patterns,  T shape=(6, 6)
SOL: 2208 patterns,  T shape=(6, 6)
XRP: 2208 patterns,  T shape=(6, 6)


In [8]:
# Visualise transition matrix heatmaps
n_perms = factorial(BEST_D)
labels  = [str(p) for p in list(itertools.permutations(range(BEST_D)))]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, coin in zip(axes.flat, COINS):
    T = transition_matrices[coin]["T"]
    sns.heatmap(T, ax=ax, cmap="Blues", xticklabels=False, yticklabels=False,
                cbar_kws={"shrink": 0.6})
    ax.set_title(f"{coin}  (d={BEST_D}, τ={BEST_TAU})", fontsize=10)
    ax.set_xlabel("Next pattern"); ax.set_ylabel("Current pattern")
fig.suptitle(f"Ordinal Pattern Transition Matrices  (d={BEST_D}, τ={BEST_TAU})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_transition_matrices.png", bbox_inches="tight")
plt.show()

## 4 · Xây Đồ Thị Ordinal Network

Mỗi node = 1 ordinal pattern; mỗi cạnh có trọng số = xác suất chuyển tiếp $T_{ij}$.

In [9]:
def build_ordinal_graph(T_matrix: np.ndarray, idx2pat: list,
                        weight_threshold: float = 0.01) -> nx.DiGraph:
    """
    Build a directed weighted graph from the transition probability matrix.
    Edges with weight < threshold are pruned to reduce noise.
    """
    G = nx.DiGraph()
    n = T_matrix.shape[0]
    for i, pat in enumerate(idx2pat):
        G.add_node(i, label=str(pat))
    for i in range(n):
        for j in range(n):
            w = T_matrix[i, j]
            if w >= weight_threshold:
                G.add_edge(i, j, weight=w)
    return G


# Build and store graphs
ordinal_graphs = {}
for coin in COINS:
    tm = transition_matrices[coin]
    G  = build_ordinal_graph(tm["T"], tm["idx2pat"])
    ordinal_graphs[coin] = G
    print(f"{coin}: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, "
          f"density={nx.density(G):.3f}")

ADA: nodes=6, edges=18, density=0.600
BNB: nodes=6, edges=18, density=0.600
BTC: nodes=6, edges=18, density=0.600
DOGE: nodes=6, edges=18, density=0.600
ETH: nodes=6, edges=18, density=0.600
LTC: nodes=6, edges=18, density=0.600
SOL: nodes=6, edges=18, density=0.600
XRP: nodes=6, edges=18, density=0.600


In [10]:
# Visualise ordinal networks (spring layout, edge weight = line width)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
cmap = cm.get_cmap("Blues")

for ax, coin in zip(axes.flat, COINS):
    G = ordinal_graphs[coin]
    pos = nx.spring_layout(G, seed=42, k=2.0)

    weights   = [G[u][v]["weight"] for u, v in G.edges()]
    edge_cols = [cmap(w * 2) for w in weights]
    widths    = [w * 6 for w in weights]

    # Node size ~ in-degree
    node_sizes = [300 + 200 * G.in_degree(n, weight="weight") for n in G.nodes()]

    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes,
                           node_color="steelblue", alpha=0.8)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_cols,
                           width=widths, arrows=True, arrowsize=10, alpha=0.7,
                           connectionstyle="arc3,rad=0.1")
    labels = {n: str(n) for n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels, ax=ax, font_size=7)
    ax.set_title(f"{coin}", fontsize=11)
    ax.axis("off")

fig.suptitle(f"Ordinal Networks  (d={BEST_D}, τ={BEST_TAU}, threshold=0.01)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_ordinal_networks.png", bbox_inches="tight")
plt.show()

## 5 · Rolling Ordinal Patterns theo thời gian (Time-varying analysis)

In [11]:
ROLL_WIN = 180   # rolling window for time-varying PE
STEP     = 30

rolling_pe_dict = {}
for coin in COINS:
    series = log_ret[coin].values
    dates  = log_ret.index
    pe_series = []
    idx_series = []

    for start in range(0, len(series) - ROLL_WIN - (BEST_D - 1) * BEST_TAU, STEP):
        window = series[start: start + ROLL_WIN]
        pats   = encode_ordinal_patterns(window, BEST_D, BEST_TAU)
        if len(pats) > 0:
            pe_series.append(permutation_entropy(pats, BEST_D))
            idx_series.append(dates[start + ROLL_WIN - 1])

    rolling_pe_dict[coin] = pd.Series(pe_series, index=idx_series)

rolling_pe_df = pd.DataFrame(rolling_pe_dict)
rolling_pe_df.index.name = "date"

fig, ax = plt.subplots(figsize=(14, 5))
for coin in COINS:
    ax.plot(rolling_pe_df.index, rolling_pe_df[coin], label=coin, linewidth=1.2)
ax.set_xlabel("Date"); ax.set_ylabel("Normalised Permutation Entropy")
ax.set_title(f"Rolling Permutation Entropy  (window={ROLL_WIN}d, d={BEST_D}, τ={BEST_TAU})",
             fontsize=12)
ax.legend(ncol=4, fontsize=9); ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_rolling_PE.png", bbox_inches="tight")
plt.show()

## 6 · Lưu kết quả

In [ ]:
# Save optimal params
opt_params = pd.DataFrame([{"d": BEST_D, "tau": BEST_TAU}])
opt_params.to_csv(DATA_DIR / "optimal_params.csv", index=False)
print(f"✓ Saved optimal_params: d={BEST_D}, τ={BEST_TAU}")

# Save grid search results
grid_df.to_csv(DATA_DIR / "gridsearch_results.csv", index=False)
print(f"✓ Saved gridsearch_results ({len(grid_df)} rows)")

# Save transition matrices (pickle)
with open(DATA_DIR / "transition_matrices.pkl", "wb") as f:
    pickle.dump(transition_matrices, f)
print("✓ Saved transition_matrices.pkl")

# Save ordinal patterns (pickle)
# Convert numpy object arrays to lists for portability
saveable_patterns = {coin: [tuple(p) for p in pats] for coin, pats in all_patterns.items()}
with open(DATA_DIR / "ordinal_patterns.pkl", "wb") as f:
    pickle.dump(saveable_patterns, f)
print("✓ Saved ordinal_patterns.pkl")

# Save rolling PE dataframe
rolling_pe_df.to_csv(DATA_DIR / "rolling_permutation_entropy.csv")
print(f"✓ Saved rolling_permutation_entropy.csv  shape={rolling_pe_df.shape}")

# Save best per-coin params
best_per_coin.to_csv(DATA_DIR / "optimal_params_per_coin.csv", index=False)
print("✓ Saved optimal_params_per_coin.csv")

print("\n=== Ordinal Network construction complete ===")

✓ Saved optimal_params: d=3, τ=1
✓ Saved gridsearch_results (128 rows)
✓ Saved transition_matrices.pkl
✓ Saved ordinal_patterns.pkl
✓ Saved rolling_permutation_entropy.csv  shape=(68, 8)
✓ Saved optimal_params_per_coin.csv

=== Ordinal Network construction complete ===


: 